# 04 · Joins: Vendas + Funcionários + Empresas 

🎯 **Objetivo:** Responder perguntas de negócio que exigem combinar dados de múltiplas tabelas usando `join`.

**Teoria:** docs/04-dataframes-catalyst-tungsten.md

Duas perguntas que só um join resolve:

1. **Quem mais vendeu?** — `vendas` tem `id_funcionario`, mas o nome está em `funcionarios`
2. **Qual setor vendeu mais em cada período?** — `setor` só existe em `empresas`, então precisamos encadear dois joins

📌 **Conceito:** Join combina linhas de duas tabelas com base em uma coluna em comum (chave). No Spark, o otimizador Catalyst escolhe automaticamente a estratégia mais eficiente (shuffle hash join, broadcast join, etc.).

---
### 🔗 Por que joins são essenciais?

Dados reais raramente estão em uma única tabela. O modelo relacional (e o Spark SQL) organiza dados em tabelas normalizadas ligadas por chaves.

**Nosso modelo:**
```
vendas ───→ funcionarios ───→ empresas
  id_funcionario    id_empresa
```

Para responder "qual setor vendeu mais?", precisamos percorrer essa corrente. Vamos começar pelo join mais simples.

### 🔤 Operações que você vai praticar

1. **Join simples** — `join(df, "chave")`, inner join por padrão
2. **Join encadeado** — múltiplos `join` em sequência para navegar 3+ tabelas
3. **Join anti** — `join(df, "chave", "left_anti")` para achar linhas **sem** correspondência
4. **`left`/`right`/`full`** — joins que preservam linhas sem correspondência, preenchendo com `NULL`
5. **Broadcast join** — `broadcast(df)` para otimizar joins com tabelas pequenas
6. **`groupBy`/`agg`/`orderBy` pós-join** — as mesmas agregações do notebook 03, agora sobre dados combinados

Vamos praticar!


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

spark

In [ ]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("../data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("../data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("../data/bronze/vendas")

✅ **Três DataFrames carregados.** Agora temos:
- `vendas` — transações com `id_funcionario`, `valor`, data
- `funcionarios` — nome, cargo, salário, `id_empresa`
- `empresas` — nome da empresa, `setor`

Vamos conectá-los!


## Ranking de funcionários por vendas

`vendas` só tem `id_funcionario` — um número sem significado para o negócio. Para ver o **nome** do funcionário, precisamos fazer um join com `funcionarios`.

⚠️ **Atenção:** O join padrão do Spark é **inner join** (só linhas que existem nas duas tabelas). Se uma venda referencia um funcionário que não existe na tabela `funcionarios`, essa venda é descartada.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as spark_sum

# Join de vendas com funcionarios pela chave id_funcionario
# Depois: agrupa por nome e cargo, soma vendas, ordena do maior para o menor
ranking_funcionarios = (
    sdf_vendas.join(sdf_funcionarios, "id_funcionario")
    .groupBy("nome_funcionario", "cargo")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
# truncate=False evita que nomes longos sejam cortados
ranking_funcionarios.show(15, truncate=False)

📌 **Interpretação do ranking:**

- Agora temos o **nome** do funcionário ao lado do total de vendas — informação que não existia em nenhuma tabela isoladamente.
- O join combinou `vendas` e `funcionarios` pela chave `id_funcionario`, pareando cada venda com os dados do vendedor.
- O `groupBy` por `nome_funcionario` e `cargo` agregou as vendas de cada pessoa.

💡 **Dica:** O Spark é inteligente: ele empurra o join para antes da agregação, minimizando o volume de dados embaralhados.

⚠️ **Pegadinha silenciosa:** repare que `vendas` **e** `funcionarios` têm, cada uma, sua própria coluna `id_empresa`. Depois deste join, o resultado carrega **duas** colunas chamadas `id_empresa` — não deu erro aqui porque não usamos essa coluna, mas `ranking_funcionarios.select("id_empresa")` lançaria `[AMBIGUOUS_REFERENCE]`. Vamos resolver isso no próximo join.


## Total de vendas por setor e período (join encadeado)

Agora a pergunta é mais ambiciosa: **qual setor da empresa vendeu mais?** O problema é que `setor` só existe em `empresas`, que se conecta a `vendas` através de `funcionarios`.

A rota é: vendas → funcionarios (via `id_funcionario`) → empresas (via `id_empresa`).

🧠 **Por quê dois joins?** Porque o modelo de dados é normalizado: cada tabela guarda apenas suas próprias colunas, e as chaves estrangeiras fazem a ligação.

⚠️ **Resolvendo a pegadinha do `id_empresa`:** como vimos acima, `vendas` já tem sua própria coluna `id_empresa`. Para o segundo join usar a `id_empresa` de `funcionarios` (a rota relacional correta) sem ambiguidade, descartamos a coluna de `vendas` **antes** de encadear.

In [ ]:
# Descarta o id_empresa de vendas ANTES do segundo join — evita a ambiguidade
# apontada acima e garante que o join usa a id_empresa de funcionarios
vendas_por_setor_encadeado = (
    sdf_vendas.drop("id_empresa")
    .join(sdf_funcionarios, "id_funcionario")
    .join(sdf_empresas, "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_encadeado.show(15)

📌 **Três tabelas, uma resposta:**

- Este resultado só foi possível porque encadeamos dois joins.
- O Spark Catalyst Optimizer funde os dois joins em um único plano de execução, buscando a estratégia mais eficiente.
- Perceba que agora temos `setor`, `ano` e `mes` na mesma linha — dados que vieram de três tabelas diferentes.

🧠 **Para refletir:** O que aconteceria se um funcionário estivesse em `vendas` mas não em `funcionarios`? Vamos responder isso já já com um `left_anti` join.


## O mesmo resultado, pelo atalho denormalizado

`vendas` já carrega `id_empresa` (o empregador do funcionário daquela venda, gravado no momento da geração dos dados) — então dá pra pular direto para `empresas`, sem passar por `funcionarios`.

Duas rotas, mesmo destino. Compare as somas de `total_vendas` entre as duas células — elas devem ser **idênticas**.

In [ ]:
from pyspark.sql.functions import broadcast

# broadcast() força o Spark a copiar a tabela empresas para TODOS os
# executores, evitando o shuffle caro da tabela vendas (que é grande).
# Só funciona porque empresas tem poucas linhas (~50).
vendas_por_setor_direto = (
    sdf_vendas.join(broadcast(sdf_empresas), "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy(col("total_vendas").desc())
)
vendas_por_setor_direto.show(15)

📌 **Comparando as duas abordagens:**

- **Join encadeado** (vendas → funcionarios → empresas): segue o modelo relacional, mas faz 2 shuffles.
- **Join direto** (vendas → empresas com `id_empresa`): apenas 1 join, e com broadcast! Ideal quando a tabela intermediária (`funcionarios`) não é necessária.

⚠️ **Atenção:** O atalho denormalizado só funciona porque o dataset já foi gerado com `id_empresa` em `vendas`. Em sistemas reais, isso nem sempre está disponível.


## Prévia: por que o segundo join foi `broadcast`?

`empresas` tem só ~50 linhas — cabe inteira na memória de cada executor. Com `broadcast()`, o Spark evita o **shuffle** (embaralhamento dos dados pela rede) e simplesmente copia a tabela pequena para todos os nós.

| Estratégia | Sem broadcast | Com broadcast |
|---|---|---|
| Shuffle? | Sim (embaralha `vendas` inteira) | Não (copia `empresas`) |
| Performance | Pode ser lenta com dados grandes | Muito mais rápida |
| Quando usar? | Tabelas grandes dos dois lados | Uma tabela é pequena |

💡 **Dica:** O Spark Catalyst Optimizer já faz broadcast automático para tabelas com menos de 10 MB (configurável via `spark.sql.autoBroadcastJoinThreshold`). O `broadcast()` explícito garante o comportamento independentemente da configuração.

📌 O próximo notebook, **05 · Por Trás dos Panos**, mostra o plano de execução por trás dessa escolha.

#### 💡 **Exemplo 1:** Integridade referencial com `left_anti` — o que o inner join esconde?

Lá no início vimos o aviso: *"se uma venda referencia um funcionário que não existe, essa venda é descartada"* silenciosamente pelo inner join. `left_anti` faz o oposto de um join normal: devolve **só** as linhas da esquerda que **não têm** correspondência na direita — perfeito para caçar esses casos.

In [ ]:
# left_anti: só as vendas SEM funcionário correspondente
vendas_orfas = sdf_vendas.join(sdf_funcionarios, "id_funcionario", "left_anti")
print(f"Vendas sem funcionário correspondente: {vendas_orfas.count():,} de {sdf_vendas.count():,}")

# left_anti na direção oposta: funcionários que nunca aparecem em vendas
funcionarios_sem_vendas = sdf_funcionarios.join(sdf_vendas, "id_funcionario", "left_anti")
print(f"Funcionários sem nenhuma venda registrada: {funcionarios_sem_vendas.count():,} de {sdf_funcionarios.count():,}")

📌 **Resultado:** zero linhas órfãs nos dois sentidos — o dataset sintético tem integridade referencial perfeita. Em dados reais isso raramente acontece; `left_anti` é a ferramenta certa para medir o tamanho do problema **antes** de decidir se um `inner join` é seguro.

#### 💡 **Exemplo 2:** Quem gera mais receita por real de salário?

Total vendido não conta a história toda — um Diretor que vende R\$27 mil pode custar muito mais que um Vendedor Junior que vende R\$13 mil. Vamos agregar as vendas **antes** do join (reduz drasticamente o volume de dados embaralhados) e depois cruzar com o salário para calcular uma métrica de eficiência.

In [ ]:
# Agrega ANTES de fazer o join — o lado esquerdo chega pequeno (1 linha por funcionário)
eficiencia_vendedores = (
    sdf_vendas.groupBy("id_funcionario")
    .agg(spark_sum("valor").alias("total_vendido"))
    .join(sdf_funcionarios, "id_funcionario")
    .withColumn("vendas_por_real_salario", col("total_vendido") / col("salario"))
    .select("nome_funcionario", "cargo", "salario", "total_vendido", "vendas_por_real_salario")
    .orderBy(col("vendas_por_real_salario").desc())
)
eficiencia_vendedores.show(10, truncate=False)

📌 **Achado de negócio:** o topo do ranking é dominado por **Vendedores Junior** — salário baixo, volume de vendas competitivo. Isso não significa que juniores "vendem mais" que diretores em valor absoluto, mas mostra um retorno por real de salário muito mais alto.

🧠 **Por quê agregar antes do join?** `sdf_vendas` tem 500 mil linhas; depois do `groupBy("id_funcionario")` sobram só ~5 mil. Fazer o join **depois** de agregar reduz drasticamente o volume de dados embaralhados.

#### 💡 **Exemplo 3:** Ticket médio por setor — nem sempre quem vende mais no total vende melhor

Já vimos o **total** de vendas por setor. Mas o setor com maior total pode simplesmente ter mais transações — não necessariamente vender **melhor** por venda. `avg("valor")` por setor responde à pergunta certa.

In [ ]:
from pyspark.sql.functions import avg

ticket_medio_por_setor = (
    sdf_vendas.drop("id_empresa")
    .join(sdf_funcionarios, "id_funcionario")
    .join(sdf_empresas, "id_empresa")
    .groupBy("setor")
    .agg(
        avg("valor").alias("ticket_medio"),
        spark_sum("valor").alias("total_vendas"),
    )
    .orderBy(col("ticket_medio").desc())
)
ticket_medio_por_setor.show(truncate=False)

📌 **Comparando as duas métricas:** `Industria` lidera em ticket médio (~R\$92) mas está longe do topo em total de vendas — sinal de menos transações, de maior valor cada. Já `Agronegocio` e `Telecomunicacoes` têm os maiores totais, com ticket médio mediano — sinal de **volume** alto compensando um ticket menor. `Financas` tem o menor ticket médio de todos os setores.

#### 💡 **Exemplo 4:** `left`, `right` e `full` — o que sobrevive quando não há correspondência?

Até aqui só vimos `inner` (mantém só o que casa dos dois lados) e `left_anti` (só o que **não** casa). Faltam os joins que **preservam** linhas sem correspondência, preenchendo com `NULL`:

- **`left`** — mantém **todas** as linhas da esquerda; preenche com `NULL` o que não casar à direita.
- **`right`** — o espelho do `left`: mantém tudo da direita. Na prática, quase ninguém usa `right` — é mais legível trocar a ordem das tabelas e escrever como `left`.
- **`full`** (ou `outer`) — mantém **tudo** dos dois lados, com `NULL` nos dois sentidos.

⚠️ Como já provamos com `left_anti`, as tabelas completas não têm órfãos — um `left`/`full` contra elas ficaria idêntico a um `inner`. Para ver `NULL`s de verdade, filtramos um dos lados de propósito, simulando um cenário realista: "só tenho o cadastro **completo** de uma parte dos funcionários".

In [ ]:
# Cenário: só temos o cadastro completo dos Diretores Comerciais
diretores = sdf_funcionarios.filter(col("cargo") == "Diretor Comercial")
print(f"Total de diretores: {diretores.count()}")

# left: preserva TODAS as vendas, preenchendo nome/cargo com NULL quando não for diretor
vendas_com_diretor = (
    sdf_vendas.join(diretores, "id_funcionario", "left")
    .select("id_venda", "id_funcionario", "valor", "nome_funcionario", "cargo")
)

print(f"Linhas no left join: {vendas_com_diretor.count():,} (= total de vendas: {sdf_vendas.count():,})")
print(f"Vendas SEM diretor correspondente: {vendas_com_diretor.filter(col('cargo').isNull()).count():,}")
vendas_com_diretor.filter(col("cargo").isNull()).show(5)

📌 O `left` manteve as 500.000 vendas — inclusive as de quem não é diretor, só que com `nome_funcionario`/`cargo` como `NULL`. Compare com um `inner join` das mesmas tabelas: ele devolveria só as 84.717 vendas feitas por diretores, descartando o resto silenciosamente.

In [ ]:
# full: cruza diretores com vendas GRANDES (> R$3000) — dos dois lados pode sobrar gente sem par
vendas_grandes = sdf_vendas.filter(col("valor") > 3000)
print(f"Total de vendas grandes: {vendas_grandes.count()}")

diretores_x_vendas_grandes = diretores.join(vendas_grandes, "id_funcionario", "full")

matched = diretores_x_vendas_grandes.filter(col("nome_funcionario").isNotNull() & col("valor").isNotNull())
so_diretor = diretores_x_vendas_grandes.filter(col("nome_funcionario").isNotNull() & col("valor").isNull())
so_venda = diretores_x_vendas_grandes.filter(col("nome_funcionario").isNull() & col("valor").isNotNull())

print(f"Diretores COM uma venda grande: {matched.count()}")
print(f"Diretores SEM nenhuma venda grande: {so_diretor.count()}")
print(f"Vendas grandes feitas por NÃO-diretores: {so_venda.count()}")

📌 O `full` respondeu três perguntas de uma vez: quem casou dos dois lados (3 diretores com venda grande), quem só existe à esquerda (846 diretores sem nenhuma venda grande) e quem só existe à direita (11 vendas grandes feitas por quem não é diretor). Nem `inner`, nem `left`, nem `left_anti` sozinhos dariam essa visão completa.

🧠 **E o `right`?** `funcionarios.join(vendas, "id_funcionario", "right")` daria exatamente o mesmo resultado que `vendas.join(funcionarios, "id_funcionario", "left")` — só com a ordem das colunas diferente. Prefira sempre reescrever como `left`: é mais fácil de ler, porque a tabela que você quer **preservar por inteiro** fica explícita à esquerda.

In [ ]:
# Encerra a SparkSession
spark.stop()

---
🎉 **Joins concluídos!** Você aprendeu:

- `join` para combinar dados de duas tabelas por uma chave (inner join por padrão)
- **Join encadeado** para navegar por relações de 3 tabelas — e como resolver colunas ambíguas com `drop()` antes de encadear
- `left_anti` para caçar linhas sem correspondência — o que o inner join descarta silenciosamente
- `left`/`right`/`full` para **preservar** linhas sem correspondência, preenchendo com `NULL` — e por que quase ninguém usa `right` na prática
- `broadcast()` para otimizar joins com tabelas pequenas
- A diferença entre seguir o modelo relacional vs. usar atalhos denormalizados
- Agregar **antes** de um join para reduzir o volume de dados embaralhados

▶️ **Próximo:** 05 · Por Trás dos Panos — lineage, DAG e o plano de execução por trás dos joins que acabamos de fazer
